## Setup

The setup cells log in to Hugging Face once for the whole session (the token is read from the Kaggle secret `HF_TOKEN`), set the input clip and the output folder, erase any results that earlier runs saved for the clip, create a fresh output folder, copy the input clip into it, and record the clip's duration and loudness envelope in `metadata.json`. Each stage installs and imports the libraries it needs, reads its inputs from `metadata.json`, and writes its results back to it. Running a stage again overwrites its previous results.

**Parameters:** `AUDIO_PATH` is the Bengali speech clip to translate; `OUTPUT_ROOT` is the folder in which the per-clip results are written.

In [ ]:
# log in to Hugging Face once, with the token stored in the Kaggle secret HF_TOKEN
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# the input clip
AUDIO_PATH = "/kaggle/input/..."
# results go to OUTPUT_ROOT/<clip name>/
OUTPUT_ROOT = "/kaggle/working/results"

In [ ]:
# libraries used by the setup cells and the shared helpers
import json
import re
import shutil
from importlib.metadata import version
from pathlib import Path

import numpy as np
import soundfile as sf
from IPython.display import Audio, display

In [ ]:
# check the input clip and create its output folder
AUDIO_PATH = Path(AUDIO_PATH)
OUTPUT_ROOT = Path(OUTPUT_ROOT)
if not AUDIO_PATH.is_file():
    raise FileNotFoundError(AUDIO_PATH)

# the clip's file name without its extension, e.g. 'original'
CLIP_ID = AUDIO_PATH.stem
OUT_DIR = OUTPUT_ROOT / CLIP_ID
if OUT_DIR.resolve() in AUDIO_PATH.resolve().parents:
    raise ValueError("AUDIO_PATH must not lie inside the clip's output folder, which is erased on every run")

# a new run on the clip erases everything that earlier runs saved for it
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True)

# every stage reads the clip from here
ORIGINAL = OUT_DIR / "original.wav"
shutil.copy2(AUDIO_PATH, ORIGINAL)

In [ ]:
# helpers shared by all stages
META_PATH = OUT_DIR / "metadata.json"


def load_meta():
    """Read metadata.json."""
    return json.loads(META_PATH.read_text(encoding="utf-8"))


def save_meta(meta):
    """Write metadata.json."""
    text = json.dumps(meta, ensure_ascii=False, indent=2)
    META_PATH.write_text(text, encoding="utf-8")


def record_env(meta, *packages):
    """Add the GPU name and the versions of the given packages to the metadata."""
    import torch

    env = meta.setdefault("environment", {})
    if torch.cuda.is_available():
        env["gpu"] = torch.cuda.get_device_name(0)
    else:
        env["gpu"] = "cpu"
    for pkg in packages:
        env[pkg] = version(pkg)


def free_gpu():
    """Release cached GPU memory after a stage has deleted its model."""
    import gc

    import torch

    gc.collect()
    torch.cuda.empty_cache()


def envelope(audio, sr, win=0.04):
    """RMS loudness in 40 ms windows, scaled so that the loudest window is 1.

    Returns a list of [window centre (s), amplitude] pairs.
    """
    n = int(win * sr)
    usable = audio[: len(audio) // n * n]
    rms = np.sqrt((usable.reshape(-1, n) ** 2).mean(axis=1))
    rms /= rms.max() + 1e-12
    return [
        [round((i + 0.5) * win, 3), round(float(a), 3)]
        for i, a in enumerate(rms)
    ]

In [ ]:
# the original clip
orig, orig_sr = sf.read(str(ORIGINAL), always_2d=True)
orig_channels = orig.shape[1]
orig = orig.mean(axis=1)

meta = {}
meta["clip_id"] = CLIP_ID
meta["source"] = str(AUDIO_PATH)
meta["original"] = {
    "duration_s": round(len(orig) / orig_sr, 3),
    "sample_rate": orig_sr,
    "channels": orig_channels,
    "envelope": envelope(orig, orig_sr),
}

save_meta(meta)
display(Audio(str(ORIGINAL)))

## 1. Transcription and Romanization

The Bengali clip is transcribed into native-script text by the transcription model. The output contains no punctuation. Each word is then romanized with `uroman`, and every character outside the aligner's vocabulary (the Latin letters a-z and the apostrophe) is removed. The romanized words are used only for alignment; later stages refer back to the native-script words.

**Input:** the input clip.

**Output:** `transcribe` entry in `metadata.json` (transcript, native words, raw and filtered romanizations).

In [ ]:
# install this stage's libraries
!pip install -q "transformers==5.17.0" huggingface_hub uroman

In [ ]:
import sys

import pandas as pd
import uroman as ur
from huggingface_hub import snapshot_download

pd.set_option("display.max_colwidth", None)

In [ ]:
meta = load_meta()

# the transcription model
ASR_MODEL = "bodhan-ai/indic-transcribe-core"
asr_model_dir = snapshot_download(ASR_MODEL)
sys.path.insert(0, asr_model_dir)

from indic_transcribe import IndicTranscribe

asr = IndicTranscribe.from_pretrained(asr_model_dir)
uroman = ur.Uroman()

# Bengali speech to Bengali text
transcription = asr(str(ORIGINAL), lang="bn")
native_words = transcription.split()

# romanize for the aligner, one romanized word per native word;
# the aligner only knows a-z and the apostrophe, so everything else is removed.
roman_raw = [uroman.romanize_string(w, lcode="ben") for w in native_words]
roman_words = [re.sub(r"[^a-z']", "", r.lower()) for r in roman_raw]

meta["transcribe"] = {
    "model": ASR_MODEL,
    "language": "bn",
    "romanizer": "uroman (lcode=ben)",
    "transcription": transcription,
    "n_words": len(native_words),
    "native_words": native_words,
    "roman_raw": roman_raw,
    "roman_words": roman_words,
}
record_env(meta, "torch", "transformers", "uroman")
save_meta(meta)

summary = {
    "transcription": transcription,
    "romanized": " ".join(roman_words),
}
display(pd.DataFrame([summary]))

del asr
free_gpu()

## 2. Forced Alignment

The audio is mixed to mono and resampled to 16 kHz, and the multilingual CTC aligner assigns a start and an end time to every romanized word. Each word also receives a confidence score, and words that are very short, overlap the previous word, follow a long silence or have a low score are flagged for review. A short padded excerpt of every word is saved so that its boundaries can be checked by ear. Words that romanize to nothing (for example stray symbols) cannot be aligned; they are left out of this and all later stages and listed as dropped.

**Input:** input clip and the romanized words from stage 1.

**Output:** `words/NN_<word>.wav` & `align` entry in `metadata.json` (per-word timings, scores and flags).

In [ ]:
import pandas as pd
import torch
import torchaudio
from IPython.display import HTML
from torchaudio.pipelines import MMS_FA as bundle

pd.set_option("display.max_colwidth", None)

In [ ]:
# inputs from stage 1
meta = load_meta()
native_words = meta["transcribe"]["native_words"]
roman_words = meta["transcribe"]["roman_words"]

# words that romanize to nothing cannot be aligned and are left out
kept = [i for i, r in enumerate(roman_words) if r]
dropped = [native_words[i] for i, r in enumerate(roman_words) if not r]

In [ ]:
# review flags and word excerpts
MIN_DUR = 0.08  # words shorter than this (s) are flagged "short"
GAP_FLAG = 0.5  # words after a silence longer than this (s) are flagged "gap"
LOW_SCORE = 0.5  # words with a confidence below this are flagged "low score"
PAD = 0.05  # padding (s) around each saved word excerpt
SR = bundle.sample_rate  # the aligner works at 16 kHz

device = "cuda" if torch.cuda.is_available() else "cpu"
alignmodel = bundle.get_model().to(device)
aligner = bundle.get_aligner()
tokenizer = bundle.get_tokenizer()

# mono 16 kHz audio
waveform, sr = torchaudio.load(str(ORIGINAL))
waveform = waveform.mean(0, keepdim=True)
waveform = torchaudio.functional.resample(waveform, sr, SR)

with torch.inference_mode():
    # frame-level character probabilities
    emission, _ = alignmodel(waveform.to(device))
    # per word: a list of token spans
    spans = aligner(emission[0], tokenizer([roman_words[k] for k in kept]))

# seconds per emission frame, and the clip length in seconds
ratio = waveform.size(1) / emission.size(1) / SR
total = waveform.size(1) / SR

(OUT_DIR / "words").mkdir(exist_ok=True)
rows = []
# starting at 0 makes the first word's "gap" the leading silence
prev_end = 0.0
for i, (k, s) in enumerate(zip(kept, spans)):
    w = roman_words[k]
    st = s[0].start * ratio
    en = s[-1].end * ratio
    # confidence, weighted by each token's length in frames
    sc = sum(t.score * len(t) for t in s) / sum(len(t) for t in s)

    flags = []
    if en - st < MIN_DUR:
        flags.append("short")
    if st < prev_end:
        flags.append("overlap")
    if st - prev_end > GAP_FLAG:
        flags.append(f"gap {st - prev_end:.1f}s")
    if sc < LOW_SCORE:
        flags.append("low score")

    # padded excerpt of the word, for checking its boundaries by ear
    excerpt = f"words/{i:02d}_{w}.wav"
    a = max(0, int((st - PAD) * SR))
    b = min(waveform.size(1), int((en + PAD) * SR))
    sf.write(OUT_DIR / excerpt, waveform[0, a:b].numpy(), SR)

    row = {
        "#": i,
        "word": w,
        "native": native_words[k],
        "start": round(st, 3),
        "end": round(en, 3),
        "dur": round(en - st, 3),
        "score": round(sc, 2),
        "flags": ", ".join(flags),
        "excerpt": excerpt,
    }
    rows.append(row)
    prev_end = en

scores = [r["score"] for r in rows]
flagged = ", ".join(f"{r['native']} ({r['flags']})" for r in rows if r["flags"])

meta["align"] = {
    "model": "torchaudio MMS_FA",
    "sample_rate": SR,
    "config": {
        "min_dur": MIN_DUR,
        "gap_flag": GAP_FLAG,
        "low_score": LOW_SCORE,
        "excerpt_pad": PAD,
    },
    "total": total,
    "speech_start": rows[0]["start"],
    "speech_end": rows[-1]["end"],
    "score_mean": round(float(np.mean(scores)), 2),
    "score_min": min(scores),
    "score_max": max(scores),
    "n_flagged": sum(bool(r["flags"]) for r in rows),
    "dropped": dropped,
    "flagged": flagged,
    "rows": rows,
}
record_env(meta, "torch", "torchaudio")
save_meta(meta)

print(f"Speech: {rows[0]['start']}-{rows[-1]['end']} s")
print(f"Confidence: mean {meta['align']['score_mean']}, range {min(scores)}-{max(scores)}")
print(f"Flagged: {flagged or 'none'}")
print(f"Dropped: {', '.join(dropped) or 'none'}")

# word table with a player for each excerpt
players = [Audio(str(OUT_DIR / r["excerpt"]))._repr_html_() for r in rows]
word_table = pd.DataFrame(rows).assign(audio=players).drop(columns="excerpt")
display(HTML(word_table.to_html(escape=False, index=False)))

del alignmodel, emission
free_gpu()

## 3. Prosody Marking

The word timings and the waveform are analyzed with fixed rules.

- **Pauses:** the silence before each word is compared with two thresholds. A gap of at least 0.20 s is marked as a short pause (`,`), and a gap of at least 0.50 s as a long pause (`|`). The silences before the first word and after the last word are not marked.
- **Loudness:** each word's level is taken as the mean of the loudest half of its 25 ms frames and compared with the rest of the clip; words well above or below the clip's typical level are labelled *loud* or *soft*.
- **Elongation:** each word's duration is divided by its romanized length, and words that are unusually slow for their length are labelled *dragged*.

The marked sentence is rebuilt with each pause marker placed before the word that follows the pause. Only the pause markers are written into the sentence; loudness and elongation are recorded for analysis.

**Input:** input clip and the word timings from stage 2.

**Output:** `transcript_marked.txt`, `roman_marked.txt` & `marking` entry in `metadata.json`.

In [ ]:
import pandas as pd
import torchaudio

pd.set_option("display.max_colwidth", None)

In [ ]:
# inputs from stage 2
meta = load_meta()
rows = meta["align"]["rows"]
total = meta["align"]["total"]
SR = meta["align"]["sample_rate"]

# the same mono 16 kHz audio that the aligner used
waveform, sr = torchaudio.load(str(ORIGINAL))
waveform = waveform.mean(0, keepdim=True)
waveform = torchaudio.functional.resample(waveform, sr, SR)

In [ ]:
# pauses
SHORT_TOK = ","
LONG_TOK = "|"
SHORT_TH = 0.20  # a gap of at least this (s) is a short pause
LONG_TH = 0.50  # a gap of at least this (s) is a long pause
MARK_EDGES = False  # also mark the silence before the first and after the last word

# loudness
EMPH_Z = 0.8  # |z| at or above this: the word is "loud" / "soft"
FRAME = 0.025  # analysis frame length (s)
HOP = 0.010  # hop between frames (s)

# elongated words
DRAG_Z = 1.0  # robust z-score at or above this: the word is "dragged"
MIN_CHARS = 2  # shorter words are skipped, as their per-character timing is noisy

# sentence markers, e.g. "^", "~", "..."
LOUD_MARK = ""
SOFT_MARK = ""
DRAG_MARK = ""

In [ ]:
# per-word data from the alignment
wav = waveform.squeeze().numpy()
N = len(rows)
words = [r["native"] for r in rows]
romans = [r["word"] for r in rows]
start = np.array([r["start"] for r in rows])
end = np.array([r["end"] for r in rows])
dur = end - start

# gap before each word: index 0 is the leading silence, and overlaps count as 0
gap_before = np.concatenate([[start[0]], start[1:] - end[:-1]])
gap_before = np.clip(gap_before, 0, None)
gap_after_last = max(total - end[-1], 0)



def pause_token(gap):
    """Pause marker for a silence of the given length."""
    if gap >= LONG_TH:
        return LONG_TOK
    if gap >= SHORT_TH:
        return SHORT_TOK
    return ""


pause = [pause_token(g) for g in gap_before]
tail = pause_token(gap_after_last)
if not MARK_EDGES:
    pause[0] = ""
    tail = ""

# every marked pause as a span of silence
# ("before" is the word it precedes, "end" is the trailing silence)
pause_spans = []
for i in range(N):
    if pause[i]:
        span_start = end[i - 1] if i > 0 else 0.0
        pause_spans.append((words[i], span_start, start[i], pause[i]))
if tail:
    pause_spans.append(("end", end[-1], total, tail))

pauses = []
for w, a, b, tok in pause_spans:
    span = {
        "before": w,
        "start": round(float(a), 3),
        "end": round(float(b), 3),
        "mid": round(float(a + b) / 2, 3),
        "length": round(float(b - a), 3),
        "type": "long" if tok == LONG_TOK else "short",
    }
    pauses.append(span)

In [ ]:
# frame levels in dB
frame_len = int(FRAME * SR)
hop_len = int(HOP * SR)
frames = np.lib.stride_tricks.sliding_window_view(wav, frame_len)[::hop_len]
frame_rms = np.sqrt((frames**2).mean(axis=1) + 1e-10)
db = 20 * np.log10(frame_rms + 1e-10)

# per word: the mean of the loudest half of its frames, so that stop-consonant dips are ignored
loud_db = []
for s, e in zip(start, end):
    a = min(int(s / HOP), len(db) - 1)
    b = min(max(int(e / HOP), a + 1), len(db))
    seg = np.sort(db[a:b])
    loud_db.append(seg[len(seg) // 2 :].mean())
loud_db = np.array(loud_db)

# words well above / below the clip's average are "loud" / "soft"
loud_z = (loud_db - loud_db.mean()) / (loud_db.std() + 1e-9)
emph = np.where(loud_z <= -EMPH_Z, "soft", "")
emph = np.where(loud_z >= EMPH_Z, "loud", emph)

In [ ]:
# seconds per romanized character, compared with the clip's median by a robust (median / MAD) z-score
nchar = np.array([len(r) for r in romans])
spc = dur / np.maximum(nchar, 1)
med = np.median(spc)
mad = np.median(np.abs(spc - med)) * 1.4826
drag_z = (spc - med) / (mad + 1e-9)

# duration relative to the duration expected for the word's length (2.0 = twice as long)
x_normal = dur / (np.maximum(nchar, 1) * med + 1e-9)
drag = (drag_z >= DRAG_Z) & (nchar >= MIN_CHARS)

In [ ]:
# markers placed before / after each word
pre = []
for e in emph:
    if e == "loud":
        pre.append(LOUD_MARK)
    elif e == "soft":
        pre.append(SOFT_MARK)
    else:
        pre.append("")
suf = [DRAG_MARK if d else "" for d in drag]

# marked sentences and the full per-word table
bn_parts = []
roman_parts = []
table = []
for i in range(N):
    marked = pre[i] + words[i] + suf[i]
    bn_parts += [pause[i], marked]
    roman_parts += [pause[i], pre[i] + romans[i] + suf[i]]
    row = {
        "#": i,
        "word": words[i],
        "roman": romans[i],
        "start": start[i],
        "end": end[i],
        "dur": round(dur[i], 3),
        "gap_before": round(float(gap_before[i]), 3),
        "pause": pause[i],
        "loud_dB": round(float(loud_db[i]), 1),
        "loud_z": round(float(loud_z[i]), 2),
        "emphasis": str(emph[i]),
        "chars": int(nchar[i]),
        "sec/char": round(float(spc[i]), 3),
        "x_normal": round(float(x_normal[i]), 2),
        "drag_z": round(float(drag_z[i]), 2),
        "drag": "DRAGGED" if drag[i] else "",
        "marked": marked,
    }
    table.append(row)
bn_parts.append(tail)
roman_parts.append(tail)

# the trailing silence gets its own row
tail_row = {
    "#": "end",
    "start": end[-1],
    "end": round(total, 3),
    "gap_before": round(float(gap_after_last), 3),
    "pause": tail,
}
table.append(tail_row)

bn_clean = " ".join(words)
bn_marked = " ".join(p for p in bn_parts if p)
roman_marked = " ".join(p for p in roman_parts if p)

meta["marking"] = {
    "config": {
        "short_tok": SHORT_TOK,
        "long_tok": LONG_TOK,
        "short_th": SHORT_TH,
        "long_th": LONG_TH,
        "mark_edges": MARK_EDGES,
        "emph_z": EMPH_Z,
        "frame": FRAME,
        "hop": HOP,
        "drag_z": DRAG_Z,
        "min_chars": MIN_CHARS,
        "loud_mark": LOUD_MARK,
        "soft_mark": SOFT_MARK,
        "drag_mark": DRAG_MARK,
    },
    "bn_clean": bn_clean,
    "bn_marked": bn_marked,
    "roman_marked": roman_marked,
    "inner_gaps": [round(float(g), 3) for g in gap_before[1:]],
    "edge_gaps": [round(float(gap_before[0]), 3), round(float(gap_after_last), 3)],
    "pauses": pauses,
    "n_loud": int((emph == "loud").sum()),
    "n_soft": int((emph == "soft").sum()),
    "n_dragged": int(drag.sum()),
    "table": table,
}
(OUT_DIR / "transcript_marked.txt").write_text(bn_marked, encoding="utf-8")
(OUT_DIR / "roman_marked.txt").write_text(roman_marked, encoding="utf-8")
save_meta(meta)

print(f"Pause thresholds: short {SHORT_TH:.2f} s, long {LONG_TH:.2f} s")
display(pd.DataFrame(table).fillna(""))
display(pd.DataFrame(pauses))

sentences = {"Bengali (marked)": bn_marked, "Roman (marked)": roman_marked}
display(pd.DataFrame([sentences]))

## 4. Translation

The translation model translates two versions of the Bengali sentence into English: the clean transcript and the pause-marked transcript. Decoding is greedy, so the same input always produces the same output.

**Input:** the clean and the pause-marked Bengali sentence from stage 3.

**Output:** `translation.txt` & `translate` entry in `metadata.json`.

In [ ]:
# install this stage's libraries
!pip install -q "transformers==5.17.0"

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

pd.set_option("display.max_colwidth", None)

In [ ]:
# inputs from stage 3
meta = load_meta()
bn_clean = meta["marking"]["bn_clean"]
bn_marked = meta["marking"]["bn_marked"]
SHORT_TOK = meta["marking"]["config"]["short_tok"]
LONG_TOK = meta["marking"]["config"]["long_tok"]

In [ ]:
MT_MODEL = "bodhan-ai/indic-translate"
PROMPT = "Translate the following text into {target_language}:\n\n{text}"
MAX_NEW_TOKENS = 512

transprocessor = AutoProcessor.from_pretrained(MT_MODEL)
transmodel, mt_info = AutoModelForImageTextToText.from_pretrained(
    MT_MODEL,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
    output_loading_info=True,
)
transmodel.eval()


def translate(text, target_language="English", max_new_tokens=MAX_NEW_TOKENS):
    """Translate text with greedy decoding."""
    prompt = PROMPT.format(target_language=target_language, text=text)
    messages = [{"role": "user", "content": prompt}]
    inputs = transprocessor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(transmodel.device)

    with torch.inference_mode():
        output = transmodel.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    # keep only the newly generated tokens
    new_tokens = output[0, inputs["input_ids"].shape[-1] :]
    return transprocessor.decode(new_tokens, skip_special_tokens=True).strip()


en_clean = translate(bn_clean)
en_marked = translate(bn_marked)

meta["translate"] = {
    "model": MT_MODEL,
    "prompt": PROMPT,
    "target_language": "English",
    "decoding": {
        "do_sample": False,
        "max_new_tokens": MAX_NEW_TOKENS,
    },
    "loading": {k: sorted(map(str, v)) for k, v in mt_info.items()},
    "en_clean": en_clean,
    "en_marked": en_marked,
    # does the translator carry the pause markers into its output?
    "markers_in": {
        "long": bn_marked.count(LONG_TOK),
        "short": bn_marked.count(SHORT_TOK),
    },
    "markers_out": {
        "long": en_marked.count(LONG_TOK),
        "short": en_marked.count(SHORT_TOK),
    },
}
(OUT_DIR / "translation.txt").write_text(en_marked, encoding="utf-8")
record_env(meta, "torch", "transformers")
save_meta(meta)

comparison = [
    {"": "clean", "Bengali": bn_clean, "English": en_clean},
    {"": "marked", "Bengali": bn_marked, "English": en_marked},
]
display(pd.DataFrame(comparison))

del transmodel, transprocessor
free_gpu()

## 5. Voice-Cloned Speech Synthesis

The marked English translation is spoken by a zero-shot voice-cloning speech synthesis model. The original clip serves as the voice reference, and the pause-marked Bengali transcript serves as the transcript of that reference. Generation samples with a fixed seed, so a run can be repeated on the same hardware.

**Input:** input clip, the marked Bengali sentence from stage 3 and the marked English translation from stage 4.

**Output:** `dubbed.wav` & `tts` entry in `metadata.json`.

In [ ]:
# install this stage's libraries
!pip install -q "transformers==5.17.0"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# inputs from stages 3 and 4
meta = load_meta()
bn_marked = meta["marking"]["bn_marked"]
en_marked = meta["translate"]["en_marked"]

# voice reference: the original clip, mono, at its own sample rate
orig, orig_sr = sf.read(str(ORIGINAL), always_2d=True)
orig = orig.mean(axis=1)

In [ ]:
TTS_MODEL = "multimodalart/higgs-audio-v3-tts-4b-transformers"
TEMPERATURE = 0.8
TOP_K = 50
SEED = 0  # fixes the sampling, so that a run can be repeated on the same hardware
sample_rate = 24_000
device = "cuda:0" if torch.cuda.is_available() else "cpu"

ttstokenizer = AutoTokenizer.from_pretrained(TTS_MODEL, trust_remote_code=True)
ttsmodel, tts_info = AutoModelForCausalLM.from_pretrained(
    TTS_MODEL,
    dtype=torch.float16,
    device_map=device,
    trust_remote_code=True,
    output_loading_info=True,
)

# weights missing from the checkpoint are randomly initialised
tts_loading = {k: sorted(map(str, v)) for k, v in tts_info.items()}
if tts_loading.get("missing_keys"):
    print("WARNING: weights missing from the checkpoint:", tts_loading["missing_keys"])

# voice reference: the original clip; reference transcript: the marked Bengali
reference_audio = torch.tensor(orig, dtype=torch.float32).to(device)
torch.manual_seed(SEED)
speech = ttsmodel.generate_speech(
    en_marked,
    ttstokenizer,
    reference_audio=reference_audio,
    reference_sample_rate=orig_sr,
    reference_text=bn_marked,
    temperature=TEMPERATURE,
    top_k=TOP_K,
)
dubbed = speech.detach().float().cpu().numpy().squeeze()
if dubbed.size == 0:
    raise RuntimeError("TTS returned empty audio")

sf.write(OUT_DIR / "dubbed.wav", dubbed, sample_rate)
dub_s = round(dubbed.size / sample_rate, 3)
orig_s = meta["original"]["duration_s"]

meta["tts"] = {
    "model": TTS_MODEL,
    "loading": tts_loading,
    "inputs": {
        "text": en_marked,
        "reference_audio": "original.wav",
        "reference_text": bn_marked,
    },
    "sampling": {
        "temperature": TEMPERATURE,
        "top_k": TOP_K,
        "seed": SEED,
    },
    "sample_rate": sample_rate,
    "duration_s": dub_s,
    # positive: the synthesized speech is shorter than the original
    "length_diff_s": round(orig_s - dub_s, 3),
    "length_change_pct": round(100 * (dub_s - orig_s) / orig_s, 1),
    "envelope": envelope(dubbed, sample_rate),
}
record_env(meta, "torch", "transformers")
save_meta(meta)
display(Audio(dubbed, rate=sample_rate))

del ttsmodel, ttstokenizer, reference_audio, speech
free_gpu()

## Results and Archive

Once every stage has completed, the main figures of the run are summarized, the original and synthesized audio are presented for listening, and the clip's output folder is compressed into a single archive.

In [ ]:
import zipfile

import pandas as pd

pd.set_option("display.max_colwidth", None)

In [ ]:
meta = load_meta()
al = meta["align"]
mk = meta["marking"]
tr = meta["translate"]
tt = meta["tts"]

speech_span = f"{al['speech_start']} - {al['speech_end']}"
confidence = f"{al['score_mean']} ({al['score_min']} - {al['score_max']})"
flagged = f"{al['n_flagged']}: {al['flagged'] or 'none'}"
thresholds = f"{mk['config']['short_th']} / {mk['config']['long_th']}"
n_long = sum(p["type"] == "long" for p in mk["pauses"])
n_short = sum(p["type"] == "short" for p in mk["pauses"])
prosody = f"{mk['n_loud']} / {mk['n_soft']} / {mk['n_dragged']}"
markers = f"{tr['markers_in']['long']} / {tr['markers_out']['long']}"
synth_duration = f"{tt['duration_s']} ({tt['length_change_pct']:+.1f}%)"
missing = ", ".join(tt["loading"].get("missing_keys", [])) or "none"

summary = [
    ("original duration (s)", meta["original"]["duration_s"]),
    ("words", meta["transcribe"]["n_words"]),
    ("speech span (s)", speech_span),
    ("confidence mean (range)", confidence),
    ("words flagged for review", flagged),
    ("pause thresholds short / long (s)", thresholds),
    ("pauses (long / short)", f"{n_long} / {n_short}"),
    ("loud / soft / dragged words", prosody),
    ("long-pause markers in Bengali / English", markers),
    ("synthesized duration (s)", synth_duration),
    ("TTS weights missing from checkpoint", missing),
    ("Bengali (marked)", mk["bn_marked"]),
    ("Roman (marked)", mk["roman_marked"]),
    ("English (clean)", tr["en_clean"]),
    ("English (marked)", tr["en_marked"]),
]
display(pd.DataFrame(summary, columns=["", CLIP_ID]))

print("Original:")
display(Audio(str(ORIGINAL)))
print("Synthesized:")
display(Audio(str(OUT_DIR / "dubbed.wav")))

archive_path = OUTPUT_ROOT.parent / f"{CLIP_ID}.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_ROOT.parent))

print(archive_path)